# `klygo.datasets` overview

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/datasets/Document_Datasets_Overview.ipynb)

The `klygo.datasets` module provides 8 high-performance, reproducible functions to inspect, partition, repartition, unpartition, merge, split, remap classes, and load YOLO-format detection datasets.

## Choose a guide

| Submodule | Notebook | Colab | Use it for | APIs |
|---|---|---|---|---|
| `info.py` | [Document_Datasets_Info.ipynb](Document_Datasets_Info.ipynb) | [Colab](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/datasets/Document_Datasets_Info.ipynb) | Inspect classes, counts, splits, and verify dataset integrity | `datasets.info` |
| `load.py` | [Document_Datasets_Load.ipynb](Document_Datasets_Load.ipynb) | [Colab](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/datasets/Document_Datasets_Load.ipynb) | Load flat dataset annotations and images into unified Detections | `datasets.load` |
| `partition.py` | [Document_Datasets_Partition.ipynb](Document_Datasets_Partition.ipynb) | [Colab](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/datasets/Document_Datasets_Partition.ipynb) | Deterministically split into train, val, and test partitions | `datasets.partition` |
| `repartition.py` | [Document_Datasets_Repartition.ipynb](Document_Datasets_Repartition.ipynb) | [Colab](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/datasets/Document_Datasets_Repartition.ipynb) | Rebuild existing partitions with new split ratios | `datasets.repartition` |
| `unpartition.py` | [Document_Datasets_Unpartition.ipynb](Document_Datasets_Unpartition.ipynb) | [Colab](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/datasets/Document_Datasets_Unpartition.ipynb) | Flatten split directory structures into flat images & labels | `datasets.unpartition` |
| `merge.py` | [Document_Datasets_Merge.ipynb](Document_Datasets_Merge.ipynb) | [Colab](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/datasets/Document_Datasets_Merge.ipynb) | Combine multiple datasets and reconcile global class IDs | `datasets.merge` |
| `split.py` | [Document_Datasets_Split.ipynb](Document_Datasets_Split.ipynb) | [Colab](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/datasets/Document_Datasets_Split.ipynb) | Split into class-based or ratio-based standalone ZIP archives | `datasets.split` |
| `remap_classes.py` | [Document_Datasets_Remap_Classes.ipynb](Document_Datasets_Remap_Classes.ipynb) | [Colab](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/datasets/Document_Datasets_Remap_Classes.ipynb) | Rename class labels or remap numeric IDs in annotations | `datasets.remap_classes` |

All dataset functions accept either directories or supported ZIP archives, and accept string paths or `PathLike` objects.


In [ ]:
!pip install --upgrade --no-cache-dir git+https://github.com/IchigoMazone/klygo.git@v2-release


## Quick start: create, inspect, and partition a sample dataset


In [ ]:
from tempfile import TemporaryDirectory
from PIL import Image
from klygo import datasets, files, media

# Create a temporary workspace for this tutorial
tutorial_workspace = TemporaryDirectory(prefix="klygo-datasets-overview-")
root = files.path(tutorial_workspace.name)

# Generate a minimal sample YOLO dataset
dataset_dir = files.join(root, "sample_dataset")
images_dir = files.join(dataset_dir, "images")
labels_dir = files.join(dataset_dir, "labels")
files.mkdir(images_dir)
files.mkdir(labels_dir)

# Save dummy images and label bounding boxes
for i, name in enumerate(["apple", "banana", "apple", "banana"]):
    class_id = 0 if name == "apple" else 1
    img_path = files.join(images_dir, f"sample_{i}.jpg")
    lbl_path = files.join(labels_dir, f"sample_{i}.txt")
    media.save(img_path, Image.new("RGB", (64, 48), color=(50 + i * 40, 100, 150)), verbose=False)
    files.save(lbl_path, [f"{class_id} 0.5 0.5 0.4 0.4"], verbose=False)

# Write YOLO data.yaml configuration
files.save(
    files.join(dataset_dir, "data.yaml"),
    ["names:", "  0: apple", "  1: banana", "nc: 2"],
    verbose=False,
)
print("Dataset created at:", dataset_dir)


## Inspect dataset with `datasets.info`


In [ ]:
# Inspect dataset structure and print a Rich table summary
stats = datasets.info(dataset_dir, format="yolo", verbose=True)
print("Classes:", stats["classes"])
print("Total images:", stats["total_images"])


## Partition into train/val/test splits with `datasets.partition`


In [ ]:
# Partition dataset into 50% train, 25% val, 25% test
partitioned = datasets.partition(
    source=dataset_dir,
    target=files.join(root, "partitioned"),
    ratios=(0.5, 0.25, 0.25),
    format="yolo",
    overwrite=True,
    verbose=False,
)
print("Partitioned output:", partitioned)
print("Partition stats:", datasets.info(partitioned))


## Remember

1. **Consistent Format Flag**: All 8 dataset operations accept `format: str = "yolo"`.
2. **Safe Workspace Output**: Operations prepare output in isolated temporary areas and atomic commits prevent corrupted states.
3. **Cross-module Links**:
   - See [Files overview](../files/Document_Files_Overview.ipynb) ([Colab](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/files/Document_Files_Overview.ipynb)) for filesystem utilities.
   - See [Archive overview](../archive/Document_Archive_Overview.ipynb) ([Colab](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/archive/Document_Archive_Overview.ipynb)) for ZIP/TAR container operations.
   - See [Outputs overview](../outputs/Document_Outputs_Overview.ipynb) ([Colab](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/outputs/Document_Outputs_Overview.ipynb)) for working with loaded `Detections`.


In [ ]:
tutorial_workspace.cleanup()
